เติม missing

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import r2_score

In [2]:
# โหลดข้อมูลจาก Excel
df = pd.read_excel("/content/data_knn_imputed.xlsx")
#df = pd.read_excel("/content/data_rf_imputed.xlsx")

In [3]:
df

,Showalter,Lifted,SWEAT,K,Cross,Vertical,TT Totals,CAPE,CIN,BRN,LCL Temp,LCL P,LCL P.1,MML θ,MML q,Thickness,PWAT
0,13.29,16.28,244.92,-27.1,5.7,19.7,25.4,0.00,0.00,0.00,276.14,783.20,314.16,296.13,6.16,5749.0,12.64
1,10.78,11.77,127.18,-22.1,10.3,19.3,29.6,0.00,0.00,0.00,281.45,839.90,319.96,295.87,8.30,5746.0,16.33
2,3.53,4.54,362.30,-14.3,19.3,19.7,39.0,15.68,-46.73,0.86,288.03,903.90,331.01,296.49,11.91,5729.0,25.68
3,4.66,5.24,200.40,-18.0,18.8,19.3,38.1,0.00,-50.25,0.00,285.99,879.04,327.88,296.74,10.71,5734.0,22.45
4,7.09,8.06,170.99,-0.7,15.3,18.3,33.6,0.00,0.00,0.00,285.51,868.80,327.84,297.24,10.51,5733.0,22.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,-4.55,-5.04,352.01,40.0,25.5,30.3,55.8,318.19,-95.93,3.38,289.19,896.00,336.27,298.43,13.00,5695.0,38.11
368,0.55,2.57,162.58,2.0,23.0,23.7,46.7,6.52,-108.75,0.17,284.68,877.22,324.07,295.55,9.87,5636.0,22.42
369,8.44,9.02,142.21,-4.7,13.7,19.7,33.4,0.00,0.00,0.00,284.29,880.49,322.46,294.84,9.55,5677.0,19.51
370,10.57,10.84,103.01,-18.9,10.1,20.1,30.2,0.00,0.00,0.00,283.37,871.98,320.92,294.70,9.08,5721.0,18.15


In [ ]:
# 1. เตรียมข้อมูล
# =====================
# เลือก numeric feature
feature_cols = ['Showalter', 'Lifted', 'SWEAT', 'K', 'Cross', 'Vertical',
                'TT Totals', 'CAPE', 'CIN', 'BRN', 'LCL Temp', 'LCL P', 'LCL P.1',
                'MML θ', 'MML q', 'Thickness', 'PWAT']
X = df[feature_cols]
# Check the actual column names in df and replace 'dBZ' with the correct target column name
# For example, if the target column is named 'Target_dBZ', you would use:
# y = df['Target_dBZ']
# Assuming the target column is the last column in the dataframe based on previous steps:
y = df.iloc[:, -1]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# ตรวจสอบความสัมพันธ์เชิงตัวเลข (Correlation)
correlation_matrix = df.corr()
print(correlation_matrix)

# วาด Heatmap แสดงความสัมพันธ์
plt.figure(figsize=(12,8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# 2. ลด Feature Correlation >0.9
# =====================
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
print("Drop due to high correlation:", to_drop)
X = X.drop(columns=to_drop)

Drop due to high correlation: ['Lifted', 'Cross', 'TT Totals', 'LCL P.1', 'MML q', 'PWAT']


In [ ]:
# 3. Scaling
# =====================
# Impute missing values using KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X)
y_imputed = imputer.fit_transform(y.values.reshape(-1, 1)) # Impute missing values in y as well

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_imputed)
y_scaled = scaler.fit_transform(y_imputed)

In [ ]:
# 4. Train/Test split
# =====================
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [ ]:
# 5. เตรียม shape สำหรับ LSTM/CNN1D
# =====================
# ต้องเป็น 3D: (samples, timesteps=1, features)
X_train_3d = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_3d  = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
# 6A. LSTM Regression
# =====================
lstm_model = Sequential()
lstm_model.add(LSTM(64, activation='relu', input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train_3d, y_train, epochs=100, batch_size=16, validation_split=0.2,
               callbacks=[EarlyStopping(monitor='val_loss', patience=10)], verbose=1)


Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0037 - val_loss: 2.9355e-04
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0044 - val_loss: 5.0234e-06
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0011 - val_loss: 5.0482e-05
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0126 - val_loss: 2.8568e-05
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0025 - val_loss: 1.2192e-04
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0040 - val_loss: 1.2602e-05
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0084 - val_loss: 1.4766e-05
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0082 - val_loss: 1.6387e-05
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0114 - val_loss: 1.8060e-05
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 9.5829e-04 - val_loss: 9.4495e-05
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0051 - val_loss: 4.7200e-05
Epoch 12/100

In [ ]:
# 6B. CNN1D Regression
# =====================
cnn_model = Sequential()
cnn_model.add(Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])))
cnn_model.add(MaxPooling1D(pool_size=1))
cnn_model.add(Flatten())
cnn_model.add(Dense(50, activation='relu'))
cnn_model.add(Dense(1))
cnn_model.compile(optimizer='adam', loss='mse')

cnn_model.fit(X_train_3d, y_train, epochs=100, batch_size=16, validation_split=0.2,
              callbacks=[EarlyStopping(monitor='val_loss', patience=10)], verbose=1)


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0039 - val_loss: 8.0697e-04
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0050 - val_loss: 2.6597e-04
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0023 - val_loss: 4.7181e-04
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0070 - val_loss: 3.7670e-04
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0015 - val_loss: 4.4938e-04
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0013 - val_loss: 1.5513e-04
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0016 - val_loss: 2.1117e-04
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0011 - val_loss: 8.4801e-05
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 5.6119e-04 - val_loss: 2.3715e-04
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.4461e-04 - val_loss: 8.0160e-05
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.8850e-04 - val_loss: 8.6316e-05
Epoch 12/100
15/

In [ ]:
# 7. ประเมินผล
# =====================
from sklearn.metrics import mean_absolute_error, mean_squared_error

# LSTM
y_pred_lstm = lstm_model.predict(X_test_3d)
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
r2_lstm = r2_score(y_test, y_pred_lstm)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


In [ ]:
# CNN1D
y_pred_cnn = cnn_model.predict(X_test_3d)
mae_cnn = mean_absolute_error(y_test, y_pred_cnn)
rmse_cnn = np.sqrt(mean_squared_error(y_test, y_pred_cnn))
r2_cnn = r2_score(y_test, y_pred_cnn)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


In [ ]:
print(f"LSTM: MAE={mae_lstm:.2f}, RMSE={rmse_lstm:.2f}, R²={r2_lstm:.2f}")
print(f"CNN1D: MAE={mae_cnn:.2f}, RMSE={rmse_cnn:.2f}, R²={r2_cnn:.2f}")

LSTM: MAE=0.01, RMSE=0.01, R²=-40.27
CNN1D: MAE=0.01, RMSE=0.01, R²=-27.03


In [ ]:
# ดู shape ของ X หลัง drop correlation
print("Shape ของ X หลังลด correlation:", X.shape)

# ดูชื่อคอลัมน์ที่เหลือ
print("Feature ที่เหลือ:", X.columns.tolist())

# ดูจำนวนคอลัมน์ที่เหลือ
print("จำนวน feature ที่เหลือ:", X.shape[1])


Shape ของ X หลังลด correlation: (372, 11)
Feature ที่เหลือ: ['Showalter', 'SWEAT', 'K', 'Vertical', 'CAPE', 'CIN', 'BRN', 'LCL Temp', 'LCL P', 'MML θ', 'Thickness']
จำนวน feature ที่เหลือ: 11


ไม่ได้เติม

In [ ]:
df = pd.read_excel("/content/new data.xlsx", sheet_name="Sheet2")

In [ ]:
# 1. เตรียมข้อมูล
# =====================
# เลือก numeric feature
feature_cols = ['Showalter', 'Lifted', 'SWEAT', 'K', 'Cross', 'Vertical',
                'TT Totals', 'CAPE', 'CIN', 'BRN', 'LCL Temp', 'LCL P', 'LCL P.1',
                'MML θ', 'MML q', 'Thickness', 'PWAT']
X = df[feature_cols]
# Check the actual column names in df and replace 'dBZ' with the correct target column name
# For example, if the target column is named 'Target_dBZ', you would use:
# y = df['Target_dBZ']
# Assuming the target column is the last column in the dataframe based on previous steps:
y = df.iloc[:, -1]

In [ ]:
# 2. ลด Feature Correlation >0.9
# =====================

# Convert columns to numeric, coercing errors
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
print("Drop due to high correlation:", to_drop)
X = X.drop(columns=to_drop)

Drop due to high correlation: ['Lifted', 'Cross', 'TT Totals', 'LCL P.1', 'MML q', 'PWAT']


/tmp/ipython-input-3968409193.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = pd.to_numeric(X[col], errors='coerce')


In [ ]:
# 3. Scaling
# =====================
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
y_scaled = scaler.fit_transform(y.values.reshape(-1,1))

In [ ]:
# 4. Train/Test split
# =====================
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [ ]:
# 5. เตรียม shape สำหรับ LSTM/CNN1D
# =====================
# ต้องเป็น 3D: (samples, timesteps=1, features)
X_train_3d = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_3d  = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
# 6A. LSTM Regression
# =====================
lstm_model = Sequential()
lstm_model.add(LSTM(64, activation='relu', input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train_3d, y_train, epochs=100, batch_size=16, validation_split=0.2,
               callbacks=[EarlyStopping(monitor='val_loss', patience=10)], verbose=1)


Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - loss: nan - val_loss: nan
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: nan - val_loss: nan
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: nan - val_loss: nan
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - val_loss: nan


In [ ]:
# 7. ประเมินผล
# =====================
from sklearn.metrics import mean_absolute_error, mean_squared_error

# LSTM
y_pred_lstm = lstm_model.predict(X_test_3d)
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
r2_lstm = r2_score(y_test, y_pred_lstm)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


ValueError: Input contains NaN.

In [ ]:
# CNN1D
y_pred_cnn = cnn_model.predict(X_test_3d)
mae_cnn = mean_absolute_error(y_test, y_pred_cnn)
rmse_cnn = np.sqrt(mean_squared_error(y_test, y_pred_cnn))
r2_cnn = r2_score(y_test, y_pred_cnn)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


ValueError: Input contains NaN.

In [ ]:
print(f"LSTM: MAE={mae_lstm:.2f}, RMSE={rmse_lstm:.2f}, R²={r2_lstm:.2f}")
print(f"CNN1D: MAE={mae_cnn:.2f}, RMSE={rmse_cnn:.2f}, R²={r2_cnn:.2f}")

LSTM: MAE=0.01, RMSE=0.01, R²=-19.84
CNN1D: MAE=0.01, RMSE=0.01, R²=-42.25


In [ ]:
print("NaNs in y_test:", np.isnan(y_test).sum())
print("NaNs in y_pred_lstm:", np.isnan(y_pred_lstm).sum())

NaNs in y_test: 0
NaNs in y_pred_lstm: 75
